# Complete code generation: process parameters, RC4 and RC5

Generate a complete SIASUN robot program from explicit process inputs, inspect its
parameter and resource manifests, and export independent **RC4** and **RC5** `.spf`
files. An optional model step generates and repairs complete programs using the
same parameters and a parser-checked reference program.

**Workflow:** process inputs → typed `ProcessSpec` → parameter bindings → complete
RC4/RC5 programs → syntax / optional Lean checks → review → artifacts.

Run this notebook in the **llm-backend environment**, which supplies
`llm_backend.process_parameters`, `treesitter4robotcode`, and
`robot_language_lean_checker`. From the llm-backend repository, launch
`uv run jupyter lab vendors/ribosome/nbs/05.application.code_generation.ipynb`
after installing the project dependencies with `uv sync`. A standalone ribosome
environment is insufficient. The default run needs no model server or database.

The examples contain illustrative values and references to already taught
positions and frames. Parameter generation binds explicit values to registers;
it does not calculate poses or discover a physical process recipe. Parser and
static checks do not establish trajectory safety or process equivalence. This
notebook writes source files; it does not execute a controller program.


## 1. Configure the notebook

Use `LLM_BACKEND_ROOT` when launching outside this repository. Enable
`RUN_LLM_GENERATION` for model requests and `RUN_LEAN_VALIDATION` for the local
static checker. For Lean, build `rc5_check` as described in
`vendors/robot-language-lean-checker/README.md`, or set the separate
`RC4_LEAN_CHECKER` / `RC5_LEAN_CHECKER` executable paths. This notebook passes the
specification's register bounds directly to the checker.

Exports go into a new run directory under `assets/code_generation` in ribosome;
set `CODEGEN_OUTPUT_DIR` to use another destination or `WRITE_ARTIFACTS=False`
to only display results. All source cells use existing library APIs and are not
nbdev module exports.


In [ ]:
# | notest
import json
import os
import sys
from copy import deepcopy
from datetime import datetime, timezone
from difflib import unified_diff
from pathlib import Path
from uuid import uuid4

import pandas as pd
from IPython.display import Code, Markdown, display

start = Path(os.getenv("LLM_BACKEND_ROOT", Path.cwd())).expanduser().resolve()
BACKEND_ROOT = next(
    (p for p in (start, *start.parents)
     if (p / "src/llm_backend/process_parameters.py").is_file()), None
)
if BACKEND_ROOT is None:
    raise FileNotFoundError("Set LLM_BACKEND_ROOT to the llm-backend checkout containing process_parameters.py")
if str(BACKEND_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT / "src"))
RIBOSOME_ROOT = BACKEND_ROOT / "vendors/ribosome"

from llm_backend.process_parameters import ProcessSpec, generate_process_parameters
from llm_backend.rc5_validation import normalize_code
from treesitter4robotcode.dialects import validate_syntax

DIALECTS = ("rc4", "rc5")
RUN_LLM_GENERATION = False
RUN_LEAN_VALIDATION = False
WRITE_ARTIFACTS = True
MAX_CORRECTIONS = 2
OUTPUT_ROOT = Path(os.getenv("CODEGEN_OUTPUT_DIR", str(RIBOSOME_ROOT / "assets/code_generation"))).expanduser().resolve()


## 2. Define process inputs and generate parameter bindings

This example performs one transfer cycle: release the gripper, approach the pick
position, move to the pick position, close the gripper, wait for its input,
approach the place position, move to the place position, release the gripper,
and return to the first approach position.

Edit values and register mappings here. `I` registers require integers; `R`
registers allow finite real numbers. Joint speed is supplied in controller joint
speed units, linear speed in controller linear speed units, acceleration in
controller units, and `WAIT T` in the selected controller's native timeout units.
Confirm these units against the selected controller documentation. An exact stop
uses `blend=None` (`CNT=0` on RC4, `FINE` on RC5).

The sample bounds below are example cell allocations, not controller limits.
Replace them with the actual cell's bank ranges. RC4 uses `IN`/`OT`; RC5 uses
`DI`/`DO`. P1–P4 and TF1/UF1 must already exist in that cell.


In [ ]:
# | notest
PROCESS_REQUEST = (
    "Perform one transfer cycle using the supplied taught points and gripper I/O. "
    "Preserve the step order, parameter values, frames, and exact stops."
)

PARAMETER_INPUTS = [
    {"name": "joint_speed", "bank": "I", "index": 3, "value": 25},
    {"name": "linear_speed", "bank": "R", "index": 3, "value": 125.5},
    {"name": "acceleration", "bank": "I", "index": 4, "value": 60},
    {"name": "grip_timeout", "bank": "I", "index": 5, "value": 500},
]

def move(motion, target, speed):
    return {"kind": "move", "motion": motion, "target": {"bank": "P", "index": target},
            "speed": {"parameter": speed}, "acceleration": {"parameter": "acceleration"},
            "blend": None}

PROCESS_STEPS = [
    {"kind": "output", "channel": 8, "value": False},
    move("MOVJ", 1, "joint_speed"),
    move("MOVL", 2, "linear_speed"),
    {"kind": "output", "channel": 8, "value": True},
    {"kind": "wait_input", "channel": 9, "value": True,
     "timeout": {"parameter": "grip_timeout"}},
    move("MOVJ", 3, "joint_speed"),
    move("MOVL", 4, "linear_speed"),
    {"kind": "output", "channel": 8, "value": False},
    move("MOVJ", 1, "joint_speed"),
]

def make_process_spec(dialect):
    input_bank, output_bank = ("IN", "OT") if dialect == "rc4" else ("DI", "DO")
    bounds = {bank: {"min": 1, "max": 100} for bank in ("I", "R", "P")}
    bounds.update({bank: {"min": 1, "max": 16}
                   for bank in ("TF", "UF", input_bank, output_bank)})
    return ProcessSpec.model_validate({
        "dialect": dialect, "parameters": deepcopy(PARAMETER_INPUTS),
        "tool_frame": 1, "user_frame": 1, "steps": deepcopy(PROCESS_STEPS),
        "register_bounds": bounds,
    })

specs = {dialect: make_process_spec(dialect) for dialect in DIALECTS}
generated = {dialect: generate_process_parameters(spec) for dialect, spec in specs.items()}
parameter_rows = [{"dialect": dialect, **binding.model_dump()}
                  for dialect, result in generated.items() for binding in result.parameters]
display(pd.DataFrame(parameter_rows))


## 3. Inspect complete RC4 and RC5 code

`generate_process_parameters` emits `NOP` through `END`, validates the selected
grammar, and rejects missing parameter references, duplicate bindings,
out-of-range resources, and unsupported operations. Named speed parameters remain
register references. ACC, CNT, and WAIT timeouts are resolved to literals where
the grammar requires them. Re-run from the inputs cell after changing a value.


In [ ]:
# | notest
for dialect, result in generated.items():
    display(Markdown(f"### {dialect.upper()}"))
    display(Code(result.code, language="text"))
    display(pd.DataFrame([resource.model_dump() for resource in result.resources]))
    print("Syntax valid:", result.validation["valid"], "| Unbounded banks:", result.unbounded_banks)

display(pd.DataFrame([
    {"feature": "Numeric register", "RC4": "R3", "RC5": "R[3]"},
    {"feature": "Joint motion", "RC4": "MOVJ P1 VJ=I3 ACC=60 CNT=0",
     "RC5": "MOVJ P[1] V = I[3] ACC = 60 FINE"},
    {"feature": "Output", "RC4": "OUT OT#8=ON", "RC5": "OUT DO[8] 1"},
    {"feature": "Wait input", "RC4": "WAIT IN#9=ON T=500", "RC5": "WAIT DI[9] = 1 T = 500"},
]))


## 4. Welding process parameters and the RC5 application profile

`welding` supplies unit-labelled register assignments for an existing cell
interface. These assignments alone do not start welding. The following example
also demonstrates RC5 native stud commands under the documented `SX322013-A2`
profile: initialization counts are derived from the steps, and `ProcessCheck`
is appended before `END`. The example R20/R21 mappings and values must be replaced
with those of the cell; the gun's `PROGNo` recipe is a separate explicit input.

RC4 supports explicit register and I/O mappings through the general process
schema; native `stud_weld` steps are rejected for RC4. This example is therefore
kept as a separate RC5 program rather than translated to RC4.


In [ ]:
# | notest
stud_spec = ProcessSpec.model_validate({
    "dialect": "rc5", "application_profile": "SX322013-A2",
    "welding": {
        "current_amperes": {"name": "weld_current", "bank": "R", "index": 20, "value": 120},
        "duration_seconds": {"name": "weld_duration", "bank": "R", "index": 21, "value": 0.25},
    },
    "steps": [
        {"kind": "stud_weld", "motion": "STUDJ", "target": {"index": 10},
         "speed": 25, "acceleration": 60, "blend": 0, "gun": 1, "program": 1},
        {"kind": "stud_weld", "motion": "STUDL", "target": {"index": 11},
         "speed": 100, "acceleration": 60, "blend": 0, "gun": 1, "program": 1, "et": 1},
    ],
    "register_bounds": {"R": {"min": 1, "max": 100}, "P": {"min": 1, "max": 100}},
})
stud_generated = generate_process_parameters(stud_spec)
display(Code(stud_generated.code, language="text"))
display(pd.DataFrame([binding.model_dump() for binding in stud_generated.parameters]))
print("Application profile:", stud_generated.application_profile)
print("Domain diagnostics:", stud_generated.domain_diagnostics)


## 5. Validate source and inspect coverage

Syntax and application-profile diagnostics always run. Optional Lean checking
uses the same register bounds as parameter generation, without changing global
register-profile settings. A missing/broken checker is reported as
`unavailable`, never as a pass. Warnings about external taught positions or
unsupported semantics remain visible; a successful static check is not an
execution or behavioral-equivalence proof.


In [ ]:
# | notest
async def check_program(source, spec):
    report = validate_syntax(source, spec.dialect)
    report["syntax_valid"] = report["valid"]
    report["semantic"] = {"status": "skipped" if RUN_LEAN_VALIDATION else "not_requested", "coverage": {}}
    lines = source.strip().splitlines()
    if (not lines or lines[0].strip() != "NOP" or lines[-1].strip() != "END"
            or sum(line.strip() == "END" for line in lines) != 1):
        report["diagnostics"].append({"kind": "input", "severity": "error",
            "code": "incomplete_program", "message": "Return one complete program starting with NOP and ending with a single END."})
    from robot_language_lean_checker.welding import diagnostics
    report["diagnostics"].extend(diagnostics(source, spec.dialect, spec.application_profile))
    report["valid"] = report["valid"] and not any(
        d.get("severity") == "error" for d in report["diagnostics"]
    )
    if not report["valid"] or not RUN_LEAN_VALIDATION:
        return report
    from llm_backend.rc5_semantics import run_checker
    from robot_language_lean_checker.source import CheckerError, MESSAGES, extract_program
    try:
        events, locations = extract_program(source, spec.dialect)
        bounds = [{"bank": bank, **limits.model_dump()}
                  for bank, limits in spec.register_bounds.items()]
        findings = await run_checker(events, bounds, spec.dialect)
        report["diagnostics"].extend(
            {**locations[item["event"]], "checker": "lean", "code": item["code"],
             "severity": item["severity"], "message": MESSAGES[item["code"]]}
            for item in findings
        )
        report["semantic"] = {"status": "checked", "coverage": {
            "events_checked": len(events), "configured_banks": list(spec.register_bounds),
            "unchecked_references": sum(f["code"] in {"unknown_bounds", "dynamic_index"} for f in findings),
            "unsupported_instructions": sum(f["code"] == "unsupported_semantics" for f in findings),
            "initialization": "source_order_advisory",
        }}
    except (CheckerError, ValueError) as exc:
        report["semantic"]["status"] = "unavailable"
        report["diagnostics"].append({"kind": "infrastructure", "severity": "error",
                                      "code": "checker_unavailable", "message": str(exc)})
    report["valid"] = report["valid"] and not any(
        d.get("severity") == "error" for d in report["diagnostics"]
    )
    return report

examples = {f"transfer_{d}": (specs[d], generated[d]) for d in DIALECTS}
examples["stud_weld_rc5"] = (stud_spec, stud_generated)
validation_reports = {}
for name, (spec, result) in examples.items():
    validation_reports[name] = await check_program(result.code, spec)

display(pd.DataFrame([
    {"example": name, "syntax_valid": report["syntax_valid"], "valid": report["valid"],
     "semantic_status": report["semantic"]["status"], "diagnostics": len(report["diagnostics"])}
    for name, report in validation_reports.items()
]))
for name, report in validation_reports.items():
    if report["diagnostics"]:
        print(name)
        display(pd.DataFrame(report["diagnostics"]))


## 6. Optional model generation with bounded repair

Set `RUN_LLM_GENERATION=True` in configuration, choose a model below, and re-run
from that cell. `generate_code_stream` supplies the selected dialect's grammar.
Each request includes the typed specification, generated parameters, and complete
reference program. The model must preserve these values and operations.

The model is called once per dialect, with at most `MAX_CORRECTIONS` additional
attempts on code errors. Infrastructure failures stop repairs. Incomplete
streams, invalid final programs, and their attempt histories remain reviewable;
only validation-passing outputs are exported as `.spf` files. A diff against the
deterministic reference exposes changes requiring review, including any changes
to behavior that syntax/static checks cannot establish.

The backend model adapter supports HTTP Ollama endpoints on localhost or an IPv4
address, and DashScope's HTTPS compatible endpoint (`DASHSCOPE_API_KEY` required).
Its configuration imports read backend `.env` defaults; no model request occurs
until this section is enabled. No MongoDB, retrieval index, or API server is needed.


In [ ]:
# | notest
MODEL_HOST = os.getenv("CODEGEN_MODEL_HOST", "http://127.0.0.1")
MODEL_PORT = int(os.getenv("CODEGEN_MODEL_PORT", "11434"))
MODEL_NAME = os.getenv("CODEGEN_MODEL", "")  # Name of a model available at MODEL_HOST.

async def generate_with_repairs(spec, reference, config):
    from llm_backend.coding_service import generate_code_stream
    from llm_backend.siasun_validation import correction_prompt

    if type(MAX_CORRECTIONS) is not int or MAX_CORRECTIONS < 0:
        raise ValueError("MAX_CORRECTIONS must be a nonnegative integer")

    request = (
        PROCESS_REQUEST + "\nGenerate one complete " + spec.dialect.upper() + " program. "
        "Preserve every supplied value, resource, frame, operation and its order. "
        "Use only the selected dialect, begin with NOP, and end with END.\n"
        "Process specification:\n" + spec.model_dump_json(indent=2) +
        "\nParameter bindings:\n" + json.dumps([p.model_dump() for p in reference.parameters], indent=2) +
        "\nParser-checked reference program:\n" + reference.code
    )
    prompt, attempts = request, []
    for attempt in range(MAX_CORRECTIONS + 1):
        chunks, stream = [], None
        try:
            stream = await generate_code_stream(prompt, [], config, inline_comments=False, dialect=spec.dialect)
            async for chunk in stream:
                chunks.append(chunk)
        except Exception as exc:
            attempts.append({"attempt": attempt + 1, "code": "".join(chunks), "error": str(exc)})
            return {"status": "generation_failed", "attempts": attempts, "code": "".join(chunks),
                    "validation": None}
        finally:
            if stream is not None:
                await stream.aclose()
        source = normalize_code("".join(chunks)).strip()
        report = await check_program(source, spec)
        attempts.append({"attempt": attempt + 1, "code": source, "validation": report})
        if report["valid"]:
            return {"status": "validated", "attempts": attempts, "code": source, "validation": report}
        if report["semantic"]["status"] == "unavailable":
            return {"status": "checker_unavailable", "attempts": attempts, "code": source, "validation": report}
        if attempt < MAX_CORRECTIONS:
            prompt = correction_prompt(request, source, report["diagnostics"], spec.dialect)
    return {"status": "invalid", "attempts": attempts, "code": source, "validation": report}

model_results = {}
if RUN_LLM_GENERATION:
    if not MODEL_NAME.strip():
        raise ValueError("Set CODEGEN_MODEL or MODEL_NAME before enabling model generation")
    from llm_backend.client_sdk.config import LLMConfig, LLMModel
    model = LLMModel(HOST=MODEL_HOST, PORT=MODEL_PORT, MODEL=MODEL_NAME)
    model_config = LLMConfig(
        LLM_EMB=model, LLM_GEN=model, LLM_CODING=model,
        FAST_API_KEY_HEADER="Authorization", FAST_API_KEY="", UVICORN_HOST="127.0.0.1",
        UVICORN_PORT="40000", CHROMADB_PATH="", CHROMA_COLLECTION_NAME="",
        IQA_TABLE_NAME="", CNV_TABLE_NAME="", TOP_K=0, THINK=False, STREAM=True,
        THINK_TAG={"start": "<think>", "end": "</think>"},
    )
    for dialect in DIALECTS:
        model_results[dialect] = await generate_with_repairs(specs[dialect], generated[dialect], model_config)
        result = model_results[dialect]
        display(Markdown(f"### Model {dialect.upper()}: {result['status']}"))
        display(Code(result["code"], language="text"))
        print("\n".join(unified_diff(generated[dialect].code.splitlines(), result["code"].splitlines(),
                                     fromfile="reference", tofile="model", lineterm="")) or "No source differences")
        if result["validation"]:
            display(pd.DataFrame(result["validation"]["diagnostics"]))
        else:
            print(result["attempts"][-1]["error"])
else:
    print("Model generation disabled; deterministic RC4/RC5 programs are available above.")


## 7. Export source, specifications, manifests, and validation reports

Each run gets its own directory. JSON files preserve the specification, resource
and parameter manifests, application profile, and validation status. Model
attempts are saved separately, including unsuccessful attempts. `.spf` files
are written only when the requested checks pass; `not_requested` explicitly
identifies omitted Lean checks. These are review artifacts, not deployed jobs.


In [ ]:
# | notest
artifact_paths = []
if WRITE_ARTIFACTS:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
    run_dir = OUTPUT_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=False)

    def save_json(name, payload):
        path = run_dir / name
        path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        artifact_paths.append(path)

    def save_source(name, source):
        path = run_dir / name
        path.write_text(source + "\n", encoding="utf-8")
        artifact_paths.append(path)

    for name, (spec, result) in examples.items():
        report = validation_reports[name]
        save_json(f"{name}.json", {"specification": spec.model_dump(mode="json"),
                                   "generated": result.model_dump(mode="json"), "validation": report})
        if report["valid"]:
            save_source(f"{name}.spf", result.code)
    for dialect, result in model_results.items():
        save_json(f"model_{dialect}.json", {"model": MODEL_NAME, "host": MODEL_HOST, "port": MODEL_PORT,
                  "specification": specs[dialect].model_dump(mode="json"), **result})
        if result["status"] == "validated" and result["validation"]["valid"]:
            save_source(f"model_{dialect}.spf", result["code"])
    save_json("run.json", {"process_request": PROCESS_REQUEST, "lean_requested": RUN_LEAN_VALIDATION,
                           "model_requested": RUN_LLM_GENERATION,
                           "files": [p.name for p in artifact_paths]})
    print("Saved run:", run_dir)
    display(pd.DataFrame({"artifact": [p.name for p in artifact_paths]}))
else:
    print("Artifact writing disabled; results remain in notebook variables.")


## Adapting the workflow

Change `PARAMETER_INPUTS`, `PROCESS_STEPS`, frame selections, and bank ranges,
then re-run from section 2. Add `welding` mappings or RC5 `stud_weld` steps only
when the cell interface and application profile are known. The supported typed
steps are `move`, `output`, `wait_input`, and RC5 `stud_weld`; broader control flow
requires another program construction path.

For manual-grounded requirements, retrieve the procedure and cited parameters
with [RAG with task objects](05.application.rag.task.ipynb), then supply the
explicit values here. This notebook itself does not retrieve manual evidence.
Review the parameter/resource tables, validation coverage, and model diffs before
using any generated source in a controller-specific workflow.
